# Experiment 4: Is the Head Dropout Distorting the Output Scale?

Companion notebook to [`README.md`](README.md) for the DAAN 570 course project (Temperature Prediction with Deep Learning).

**Hypothesis (H1)**: the Transformer's error is not noise, it is a systematic compression of the output, and the `Dropout` in the dense head causes it.

exp03's 241,504 saved test predictions fit `pred = 0.875 * actual + 0.336` at correlation 0.9935, with a mean error of -1.667 °C. 46% of the test MSE (6.039) is pure squared bias, and undoing that affine distortion by hand drops test MAE from 2.043 to 0.923. Re-scoring the saved exp03 model **with dropout off** gives MSE 6.011, bias -1.700 and slope 0.875 on the *training* split, against the 1.610 Keras reported during training -- so the compression is already present on the data the model was fitted to, and it is not a generalisation failure.

The head is `GlobalAveragePooling1D -> Dense(32, relu) -> Dropout(0.20) -> Dense(16, relu) -> Dense(1)`. Inverted dropout preserves the *mean* entering `Dense(16)` but inflates its *variance* during training; because ReLU is convex, `E[relu(noisy)] > relu(E[clean])`. More signal reaches the output during training than at inference, so the output weights are calibrated to the inflated regime and the prediction comes out multiplicatively too small.

**Prediction if H1 is true**: setting the head dropout to zero, changing nothing else, removes the bias and brings the slope to roughly 1.0.

**What changed vs. exp03**: one value -- `CONFIG = TransformerConfig(head_dropout_rate=0.0)`. The four dropout layers inside the encoder blocks keep `dropout_rate=0.20`. Everything else is exp03's, unchanged.

**The control**: exp03 is rerun here alongside exp04 rather than compared against its July numbers. Both entrypoints begin with `tf.keras.utils.set_random_seed(42)`. Without a matched control, any exp04-vs-exp03 difference would be confounded with run-to-run variation.

## Setup

**Running on Colab**: this cell clones the (private) repo, installs dependencies, and pulls the Git-LFS-tracked raw data. One-time setup: create a fine-grained, read-only GitHub PAT for this repo and store it in Colab's Secrets manager (key icon in the left sidebar) under the name `GITHUB_PAT`, then grant this notebook access to it when prompted. Also select **Runtime > Change runtime type > GPU** before running.

**Running locally**: this cell detects that it's not on Colab and does nothing -- it assumes you already have the repo cloned, dependencies installed (`pip install -r requirements.txt`), and `data/splits/` available (or `data/raw/` present, from which splits regenerate automatically).

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = "exp04_head_dropout"


def run(cmd):
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")


try:
    import google.colab
    from google.colab import userdata

    REPO_DIR = Path("/content/repo")
    token = userdata.get("GITHUB_PAT")
    repo_url = f"https://{token}@github.com/LarryGreen-alt/Temperature_Prediction_DeepLearning.git"

    # A directory can be left behind by a previous failed attempt without
    # being a real git checkout -- only trust it if .git is actually there.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        shutil.rmtree(REPO_DIR)

    if (REPO_DIR / ".git").is_dir():
        run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
        run(f"git -C {REPO_DIR} checkout -q {BRANCH}")
        run(f"git -C {REPO_DIR} pull -q origin {BRANCH}")
    else:
        run(f"git clone -q -b {BRANCH} {repo_url} {REPO_DIR}")

    get_ipython().run_line_magic("cd", str(REPO_DIR))
    run("pip install -q -r requirements.txt")
    run("apt-get -qq install -y git-lfs")
    # --force: the git-lfs apt package's post-install step already sets up
    # this same hook globally, so a plain `install` refuses as a safety
    # check against clobbering a *different*, unrelated pre-push hook.
    run("git lfs install --force")
    run("git lfs pull")

    PROJECT_ROOT = REPO_DIR
except ImportError:

    def find_project_root(marker="weather_main.py"):
        for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if (candidate / marker).exists():
                return candidate
        raise FileNotFoundError(f"Could not find project root (looking for {marker})")

    PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd

from src.models.common import data
from src.models.common.compare import compare, latest_experiment_with_metrics
from src.models.common.plotting import plot_loss_curve, plot_mae_curve, plot_prediction_curve
from src.models.transformer.training import load_cached_results
from src.models.transformer.experiments.exp04_head_dropout.config import (
    CONFIG, EXPERIMENT_NAME, FEATURE_COLUMNS, CITY_EMBED_DIM
)
from src.models.transformer.experiments.exp03_city_and_temperature_features.config import (
    CONFIG as CONTROL_CONFIG, EXPERIMENT_NAME as CONTROL_NAME
)

## Data

Same source as every earlier experiment: `data/splits/{train,dev,test}.csv`, derived from `data/raw/*.csv` via `src/data/preprocess.py` -> `src/data/split_dataset.py` (chronological 70/15/15 split per city). `data.load_splits()` regenerates `data/splits/` automatically if missing.

In [ ]:
train_df, dev_df, test_df = data.load_splits()

print("Train rows:", len(train_df))
print("Dev rows  :", len(dev_df))
print("Test rows :", len(test_df))
train_df.head()

## Configuration

The control and the experiment differ in exactly one field, `head_dropout_rate`. `None` means "use `dropout_rate`", which is what every run before exp04 did.

In [ ]:
print("Control   (exp03):", CONTROL_CONFIG)
print()
print("Experiment(exp04):", CONFIG)
print()
differences = {
    key: (getattr(CONTROL_CONFIG, key), getattr(CONFIG, key))
    for key in CONFIG.to_dict()
    if getattr(CONTROL_CONFIG, key) != getattr(CONFIG, key)
}
print("Differences (control -> experiment):", differences)
print()
print("Feature columns:", FEATURE_COLUMNS)
print("City embed dim: ", CITY_EMBED_DIM)

## Run the control and the experiment

Both are launched as separate processes rather than in this kernel, so that each one starts from the same fresh RNG state: `tf.keras.utils.set_random_seed(42)` is the first executable line of each `train.py`. Running them in a single kernel would leave the second run starting from whatever state the first left behind, which is exactly the run-to-run variation the control exists to rule out.

Each takes roughly 40-60 minutes on a T4. Run the control first.

In [ ]:
!python -m src.models.transformer.experiments.exp03_city_and_temperature_features.train

In [ ]:
!python -m src.models.transformer.experiments.exp04_head_dropout.train

## Results

In [ ]:
control_dir = latest_experiment_with_metrics(f"Transformer/{CONTROL_NAME}")
experiment_dir = latest_experiment_with_metrics(f"Transformer/{EXPERIMENT_NAME}")

print("Control   dir:", control_dir)
print("Experiment dir:", experiment_dir)

results = load_cached_results(experiment_dir)

print()
print(f"Test Loss : {results['loss']:.4f}")
print(f"Test MAE  : {results['mae']:.4f}")
print(f"Test RMSE : {results['rmse']:.4f}")
print(f"Epochs trained: {results['epochs_trained']}")

figure_dir = experiment_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plot_loss_curve(results["history"], figure_dir, show=True)
plot_mae_curve(results["history"], figure_dir, show=True)
plot_prediction_curve(results["y_test"], results["predictions"], figure_dir, show=True)

pd.read_csv(experiment_dir / "predictions.csv").head(10)

## Verdict on H1

`check_exp04.py` reads `predictions.csv` from the newest exp03 and exp04 experiment directories and reports n, MAE, RMSE, bias, slope, intercept and correlation for each. H1 is accepted only if **both** primary criteria hold against the control: `abs(bias) < 0.5` °C and `slope > 0.95`. Test MAE below 1.5 is a secondary reference point, and the correlation staying above 0.99 is a guardrail -- if accuracy improves while correlation drops, the model traded structure for calibration and the result needs a closer look before any claim is made.

In [ ]:
!python check_exp04.py

## Comparison to the earlier experiments

Reuses `src/models/common/compare.py`'s `compare()`, passing an explicit list of model names so it can look inside each experiment's namespaced results directory. Note that this table reports loss/MAE/RMSE only -- the bias and slope that H1 is actually about are in the cell above.

In [ ]:
compare([
    "Transformer/baseline",
    "Transformer/exp01_temperature_feature",
    "Transformer/exp02_city_embedding",
    f"Transformer/{CONTROL_NAME}",
    f"Transformer/{EXPERIMENT_NAME}",
])

## Save results

**If running on Colab**, run the cell below to zip both runs' output folders together with this executed notebook (so the professor can see the real run, including these plots and outputs) and download it as one file. Unzip it directly into your local repo checkout, review with `git status`/`git diff`, then commit and push from your own machine.

**If running locally**, your results are already sitting in the repo.

In [ ]:
try:
    import google.colab
    from google.colab import files

    zip_name = f"{EXPERIMENT_NAME}_results.zip"
    notebook_path = f"src/models/transformer/experiments/{EXPERIMENT_NAME}/experiment.ipynb"

    run(f"zip -r {zip_name} "
        f"models/Transformer/{EXPERIMENT_NAME} "
        f"models/Transformer/{CONTROL_NAME} "
        f"{notebook_path}")
    files.download(zip_name)
except ImportError:
    print("Not running on Colab -- results are already in your local repo checkout.")